# Adversarial ML / AI Security — dev log

Módulo `core/adversarial_ml/`. Ataques adversariais reais (FGSM, PGD, model extraction, data poisoning), defesas (adversarial training, pré-processamento, detecção) e o **MODEL SECURITY REPORT** consolidado.

`nada é simulado`: as acurácias limpa/adversarial vêm da avaliação real do modelo sobre os dados perturbados.

## 1. Dataset difícil (não linearmente separável)

Duas luas entrelaçadas com ruído — um modelo linear aqui tem fronteira frágil, então o ataque tem efeito visível.

In [ ]:
import numpy as np
from core.adversarial_ml import run_security_assessment, LogisticRegressionModel
from core.adversarial_ml.robustness.metrics import robustness_curve

def two_moons(n=600, noise=0.25, seed=7):
    rng = np.random.default_rng(seed)
    t = rng.uniform(0, np.pi, n // 2)
    x0 = np.c_[np.cos(t), np.sin(t)]
    x1 = np.c_[1 - np.cos(t), 1 - np.sin(t) - 0.5]
    X = np.vstack([x0, x1]) + rng.normal(0, noise, (n, 2))
    y = np.array([0] * (n // 2) + [1] * (n // 2))
    p = rng.permutation(n)
    return X[p], y[p]

X, y = two_moons()
s = int(0.7 * len(X))
Xtr, ytr, Xte, yte = X[:s], y[:s], X[s:], y[s:]
model = LogisticRegressionModel(2, 2).fit(Xtr, ytr, lr=0.2, epochs=500)
print('clean acc:', (model.predict(Xte) == yte).mean())

## 2. Curva de robustez (acurácia adversarial × epsilon)

In [ ]:
curve = robustness_curve(model, Xte, yte, [0.0, 0.05, 0.1, 0.2, 0.3, 0.5])
for pt in curve:
    print(f"eps={pt['epsilon']:<5} adv_acc={pt['adversarial_accuracy']:.3f}")

## 3. Assessment completo + MODEL SECURITY REPORT

Inclui FGSM, PGD, model extraction, data poisoning, adversarial training como defesa e a seção de LLM security (motor real de `prompt_security` via `red_team_lab`).

In [ ]:
report = run_security_assessment(
    model, Xte, yte, n_classes=2,
    model_name='two-moons-logreg',
    epsilon=0.3,
    X_train=Xtr, y_train=ytr,
    include_llm_security=True,
    include_defense=True,
)
print(report.rendered_report)

## 4. Efeito do adversarial training

`report.defenses[0]` compara a acurácia robusta (sob PGD) antes e depois de treinar o modelo com o min-max de Madry.

In [ ]:
d = report.defenses[0]
print(d.summary)
print('risco geral:', report.overall_risk)
print('recomendações:', report.recommendations)

## 5. Uso como gate de MLOps

O `ModelSecurityReport` é o objeto que o Argus consome no `ml-platform/adversarial-evaluation/` como *production robustness gate*: um modelo só é promovido se `overall_risk` estiver dentro do limite configurado e a acurácia robusta acima do threshold.